# LoRA Tester

Test trained LoRA files against SDXL models. Supports quick generation,
LoRA scale comparison grids, and scheduler comparison grids.

**Requirements:**
- Google Colab with GPU runtime (L4 recommended)
- Trained LoRA `.safetensors` file in Google Drive
- Optionally, a custom SDXL model in Google Drive

In [ ]:
#@title 1. Install Dependencies
#@markdown Install diffusers, compel, and other required packages.

import subprocess, sys, os

gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          capture_output=True, text=True)
print(f"GPU: {gpu_info.stdout.strip()}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib
!pip install -q diffusers>=0.25.1 transformers>=4.36.2 accelerate>=0.25.0 safetensors
!pip install -q compel

print("\nInstallation complete!")

In [ ]:
#@title 2. Connect to Google Drive (API Method)
#@markdown Connect to Google Drive using the Drive API.
#@markdown
#@markdown **Setup:** Add these secrets in Colab (key icon in left sidebar):
#@markdown - `GOOGLE_CLIENT_ID` - your OAuth 2.0 client ID
#@markdown - `GOOGLE_CLIENT_SECRET` - your OAuth 2.0 client secret

import os, io, json
from google.colab import userdata
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

SCOPES = ['https://www.googleapis.com/auth/drive']

CLIENT_CREDENTIALS = {
    "installed": {
        "client_id": userdata.get('GOOGLE_CLIENT_ID'),
        "project_id": "prompt-generator-450418",
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "auth_provider_x509_cert_url": "https://www.googleapis.com/oauth2/v1/certs",
        "client_secret": userdata.get('GOOGLE_CLIENT_SECRET'),
        "redirect_uris": ["urn:ietf:wg:oauth:2.0:oob"]
    }
}

def authenticate_drive():
    creds = None
    if os.path.exists('/content/token.json'):
        creds = Credentials.from_authorized_user_file('/content/token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            creds_path = '/content/credentials.json'
            with open(creds_path, 'w') as f:
                json.dump(CLIENT_CREDENTIALS, f)
            flow = Flow.from_client_secrets_file(creds_path, scopes=SCOPES,
                                                 redirect_uri='urn:ietf:wg:oauth:2.0:oob')
            auth_url, _ = flow.authorization_url(prompt='consent')
            print(f"\n1. Click this link to authorize:\n\n{auth_url}\n")
            print("2. Sign in and click 'Allow'")
            print("3. Copy the authorization code and paste it below:\n")
            code = input("Authorization code: ").strip()
            flow.fetch_token(code=code)
            creds = flow.credentials
            os.remove(creds_path)
        with open('/content/token.json', 'w') as token:
            token.write(creds.to_json())
    return build('drive', 'v3', credentials=creds)

def get_file_id_from_path(service, path):
    path = path.replace('/content/drive/MyDrive/', '').replace('My Drive/', '').lstrip('/')
    parts = path.split('/')
    parent_id = 'root'
    for part in parts:
        query = f"name='{part}' and '{parent_id}' in parents and trashed=false"
        results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
        files_list = results.get('files', [])
        if not files_list:
            return None
        parent_id = files_list[0]['id']
    return parent_id

def download_folder(service, drive_path, local_path):
    folder_id = get_file_id_from_path(service, drive_path)
    if not folder_id:
        print(f"ERROR: Folder not found in Drive: {drive_path}")
        return False
    os.makedirs(local_path, exist_ok=True)
    query = f"'{folder_id}' in parents and trashed=false"
    results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    files_list = results.get('files', [])
    print(f"Downloading {len(files_list)} files from Drive...")
    for file in files_list:
        file_path = os.path.join(local_path, file['name'])
        if file['mimeType'] == 'application/vnd.google-apps.folder':
            download_folder(service, f"{drive_path}/{file['name']}", file_path)
        else:
            request = service.files().get_media(fileId=file['id'])
            with open(file_path, 'wb') as f:
                downloader = MediaIoBaseDownload(f, request)
                done = False
                while not done:
                    status, done = downloader.next_chunk()
            print(f"  Downloaded: {file['name']}")
    return True

def download_file(service, drive_path, local_path):
    """Download a single file from Google Drive."""
    file_id = get_file_id_from_path(service, drive_path)
    if not file_id:
        print(f"ERROR: File not found in Drive: {drive_path}")
        return False
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with open(local_path, 'wb') as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"  Download progress: {int(status.progress() * 100)}%")
    print(f"  Downloaded: {os.path.basename(local_path)}")
    return True

print("Authenticating with Google Drive API...")
drive_service = authenticate_drive()
if drive_service:
    print("Google Drive connected successfully!")

In [ ]:
#@title 3. Configuration
#@markdown ### Model
#@markdown Choose `base_sdxl` or `drive_model` for a custom SDXL model from Drive.
MODEL_SOURCE = "base_sdxl" #@param ["base_sdxl", "drive_model"]
DRIVE_MODEL_PATH = "AI/models/John6666/wai-ani-nsfw-ponyxl-v11-sdxl" #@param {type:"string"}

#@markdown ### LoRA
LORA_DRIVE_PATH = "Loras/stoo_tee/output/stoo_tee.safetensors" #@param {type:"string"}
LORA_SCALE = 0.8 #@param {type:"slider", min:0.1, max:1.5, step:0.1}
TRIGGER_WORD = "stoo_tee" #@param {type:"string"}
PREPEND_TRIGGER = True #@param {type:"boolean"}

import os

# Download LoRA from Drive
LORA_LOCAL_PATH = f"/content/lora/{os.path.basename(LORA_DRIVE_PATH)}"
print(f"Downloading LoRA from Drive: {LORA_DRIVE_PATH}")
download_file(drive_service, LORA_DRIVE_PATH, LORA_LOCAL_PATH)
lora_size_mb = os.path.getsize(LORA_LOCAL_PATH) / (1024 * 1024)
print(f"LoRA file: {LORA_LOCAL_PATH} ({lora_size_mb:.1f} MB)")

# Determine model path
if MODEL_SOURCE == "drive_model" and DRIVE_MODEL_PATH:
    MODEL_PATH = "/content/model"
    print(f"\nDownloading model from Drive: {DRIVE_MODEL_PATH}")
    download_folder(drive_service, DRIVE_MODEL_PATH, MODEL_PATH)
    print(f"Model downloaded to {MODEL_PATH}")
else:
    MODEL_PATH = "stabilityai/stable-diffusion-xl-base-1.0"

print(f"\nConfiguration:")
print(f"  Model: {MODEL_PATH}")
print(f"  LoRA: {LORA_LOCAL_PATH}")
print(f"  Scale: {LORA_SCALE}")
print(f"  Trigger: {TRIGGER_WORD}")
print(f"  Prepend trigger: {PREPEND_TRIGGER}")

In [ ]:
#@title 4. Load Pipeline + LoRA
#@markdown Loads the SDXL model, applies the LoRA, and sets up schedulers.

import torch
from diffusers import (
    StableDiffusionXLPipeline,
    DPMSolverMultistepScheduler,
    DPMSolverSinglestepScheduler,
    EulerDiscreteScheduler,
    EulerAncestralDiscreteScheduler,
    DDIMScheduler,
    UniPCMultistepScheduler,
)
from diffusers import AutoencoderKL
from compel import Compel, ReturnedEmbeddingsType

# Load pipeline
print("Loading SDXL pipeline...")
load_kwargs = dict(torch_dtype=torch.float16, use_safetensors=True)

# Try fp16 variant first, fall back to default
try:
    pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_PATH, variant="fp16", **load_kwargs)
except Exception:
    pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_PATH, **load_kwargs)

# Load enhanced VAE for better quality
print("Loading enhanced VAE...")
vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe.vae = vae

pipe = pipe.to("cuda")

# Memory optimizations
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
try:
    pipe.enable_xformers_memory_efficient_attention()
    print("Using xformers attention")
except Exception:
    from diffusers.models.attention_processor import AttnProcessor2_0
    pipe.unet.set_attn_processor(AttnProcessor2_0())
    print("Using PyTorch SDPA attention")

# Load LoRA as adapter (NOT fused — scale is applied at inference time
# via cross_attention_kwargs, which avoids floating point drift from
# repeated fuse/unfuse cycles)
print(f"Loading LoRA: {LORA_LOCAL_PATH}")
pipe.load_lora_weights(LORA_LOCAL_PATH)
print(f"LoRA loaded (scale will be applied at inference time)")

# Store base scheduler config for switching later
base_scheduler_config = dict(pipe.scheduler.config)
# Remove non-standard keys that cause errors with some schedulers
clean_scheduler_config = {k: v for k, v in base_scheduler_config.items()
                          if k not in ('skip_prk_steps', 'set_alpha_to_one',
                                       'solver_type', 'lower_order_final',
                                       'algorithm_type', 'final_sigmas_type',
                                       'solver_order')}

# Available schedulers
SAMPLERS = {
    "DPM++ 2M Karras": lambda: DPMSolverMultistepScheduler.from_config(clean_scheduler_config, use_karras_sigmas=True),
    "DPM++ 2M": lambda: DPMSolverMultistepScheduler.from_config(clean_scheduler_config),
    "DPM++ SDE Karras": lambda: DPMSolverSinglestepScheduler.from_config(clean_scheduler_config, use_karras_sigmas=True),
    "Euler": lambda: EulerDiscreteScheduler.from_config(clean_scheduler_config),
    "Euler a": lambda: EulerAncestralDiscreteScheduler.from_config(clean_scheduler_config),
    "DDIM": lambda: DDIMScheduler.from_config(clean_scheduler_config),
    "UniPC": lambda: UniPCMultistepScheduler.from_config(clean_scheduler_config),
}

# Set up Compel for SDXL prompt weighting (dual text encoder)
compel = Compel(
    tokenizer=[pipe.tokenizer, pipe.tokenizer_2],
    text_encoder=[pipe.text_encoder, pipe.text_encoder_2],
    returned_embeddings_type=ReturnedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED,
    requires_pooled=[False, True],
)

print(f"\nPipeline ready!")
print(f"Available schedulers: {', '.join(SAMPLERS.keys())}")

In [ ]:
#@title 5. Quick Generate
#@markdown Generate images with your LoRA. Include your trigger word in the prompt!

import matplotlib.pyplot as plt
from datetime import datetime

#@markdown ### Prompt
PROMPT = "stoo_tee person, professional headshot, studio lighting, neutral background" #@param {type:"string"}
NEGATIVE_PROMPT = "deformed, ugly, bad anatomy, disfigured, poorly drawn face, mutation, extra limb, bad hands, fused fingers, too many fingers, blurry, low quality" #@param {type:"string"}

#@markdown ### Generation Settings
NUM_IMAGES = 4 #@param {type:"slider", min:1, max:8, step:1}
WIDTH = 1024 #@param {type:"slider", min:512, max:1536, step:64}
HEIGHT = 1024 #@param {type:"slider", min:512, max:1536, step:64}
GUIDANCE_SCALE = 7.5 #@param {type:"slider", min:1, max:20, step:0.5}
NUM_STEPS = 30 #@param {type:"slider", min:10, max:50, step:5}
SEED = -1 #@param {type:"integer"}
SAMPLER = "DPM++ 2M Karras" #@param ["DPM++ 2M Karras", "DPM++ 2M", "DPM++ SDE Karras", "Euler", "Euler a", "DDIM", "UniPC"]

# Apply scheduler
pipe.scheduler = SAMPLERS[SAMPLER]()

# Build prompt
prompt = PROMPT
if PREPEND_TRIGGER and TRIGGER_WORD and not prompt.lower().startswith(TRIGGER_WORD.lower()):
    prompt = f"{TRIGGER_WORD}, {prompt}"

print(f"Prompt: {prompt}")
print(f"Sampler: {SAMPLER} | LoRA scale: {LORA_SCALE}")
print(f"Generating {NUM_IMAGES} images...\n")

# Generate with Compel prompt weighting
conditioning, pooled = compel(prompt)
neg_conditioning, neg_pooled = compel(NEGATIVE_PROMPT)

images = []
for i in range(NUM_IMAGES):
    seed = SEED if SEED >= 0 else torch.randint(0, 2**32, (1,)).item()
    generator = torch.Generator(device="cuda").manual_seed(seed)

    image = pipe(
        prompt_embeds=conditioning,
        pooled_prompt_embeds=pooled,
        negative_prompt_embeds=neg_conditioning,
        negative_pooled_prompt_embeds=neg_pooled,
        width=WIDTH,
        height=HEIGHT,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_STEPS,
        generator=generator,
        cross_attention_kwargs={"scale": LORA_SCALE},
    ).images[0]

    images.append((image, seed))
    print(f"  Image {i+1}/{NUM_IMAGES} (seed: {seed})")

# Display grid
cols = min(NUM_IMAGES, 4)
rows = (NUM_IMAGES + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if NUM_IMAGES == 1:
    axes = [[axes]]
elif rows == 1:
    axes = [axes] if NUM_IMAGES > 1 else [[axes]]
elif cols == 1:
    axes = [[ax] for ax in axes]

for idx, (image, seed) in enumerate(images):
    row, col = idx // cols, idx % cols
    axes[row][col].imshow(image)
    axes[row][col].set_title(f"Seed: {seed}", fontsize=10)
    axes[row][col].axis('off')
for idx in range(NUM_IMAGES, rows * cols):
    row, col = idx // cols, idx % cols
    axes[row][col].axis('off')
plt.suptitle(f"{SAMPLER} | scale={LORA_SCALE} | steps={NUM_STEPS} | cfg={GUIDANCE_SCALE}", fontsize=12)
plt.tight_layout()
plt.show()

# Save images
output_dir = "/content/test_output"
os.makedirs(output_dir, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
for idx, (image, seed) in enumerate(images):
    filepath = os.path.join(output_dir, f"{timestamp}_{idx:02d}_seed{seed}.png")
    image.save(filepath)
print(f"\nSaved {len(images)} images to {output_dir}")

In [ ]:
#@title 6. LoRA Scale Comparison
#@markdown Generate the same image at different LoRA scales to find the sweet spot.
#@markdown Uses a fixed seed so differences come only from the scale change.

import matplotlib.pyplot as plt

#@markdown ### Prompt
SCALE_PROMPT = "stoo_tee person, professional headshot, studio lighting, neutral background" #@param {type:"string"}
SCALE_NEGATIVE = "deformed, ugly, bad anatomy, disfigured, poorly drawn face, mutation, extra limb, bad hands, fused fingers, too many fingers, blurry, low quality" #@param {type:"string"}

#@markdown ### Settings
SCALE_SEED = 42 #@param {type:"integer"}
SCALE_STEPS = 30 #@param {type:"slider", min:10, max:50, step:5}
SCALE_CFG = 7.5 #@param {type:"slider", min:1, max:20, step:0.5}
SCALE_WIDTH = 1024 #@param {type:"slider", min:512, max:1536, step:64}
SCALE_HEIGHT = 1024 #@param {type:"slider", min:512, max:1536, step:64}
SCALE_SAMPLER = "DPM++ 2M Karras" #@param ["DPM++ 2M Karras", "DPM++ 2M", "DPM++ SDE Karras", "Euler", "Euler a", "DDIM", "UniPC"]

scales = [0.0, 0.4, 0.6, 0.8, 1.0, 1.2]

# Build prompt
prompt = SCALE_PROMPT
if PREPEND_TRIGGER and TRIGGER_WORD and not prompt.lower().startswith(TRIGGER_WORD.lower()):
    prompt = f"{TRIGGER_WORD}, {prompt}"

pipe.scheduler = SAMPLERS[SCALE_SAMPLER]()

# Compel embeddings
conditioning, pooled = compel(prompt)
neg_conditioning, neg_pooled = compel(SCALE_NEGATIVE)

print(f"Prompt: {prompt}")
print(f"Seed: {SCALE_SEED}")
print(f"Generating {len(scales)} images at scales: {scales}\n")

scale_images = []
for scale in scales:
    generator = torch.Generator(device="cuda").manual_seed(SCALE_SEED)

    image = pipe(
        prompt_embeds=conditioning,
        pooled_prompt_embeds=pooled,
        negative_prompt_embeds=neg_conditioning,
        negative_pooled_prompt_embeds=neg_pooled,
        width=SCALE_WIDTH,
        height=SCALE_HEIGHT,
        guidance_scale=SCALE_CFG,
        num_inference_steps=SCALE_STEPS,
        generator=generator,
        cross_attention_kwargs={"scale": scale},
    ).images[0]

    scale_images.append((image, scale))
    print(f"  Scale {scale} done")

# Display comparison grid
cols = len(scales)
fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
if cols == 1:
    axes = [axes]

for idx, (image, scale) in enumerate(scale_images):
    axes[idx].imshow(image)
    axes[idx].set_title(f"Scale: {scale}", fontsize=11, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle(f"LoRA Scale Comparison | seed={SCALE_SEED} | {SCALE_SAMPLER}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
#@title 7. Scheduler Comparison
#@markdown Generate the same image with different schedulers to compare quality.
#@markdown Uses a fixed seed and LoRA scale.

import matplotlib.pyplot as plt

#@markdown ### Prompt
SCHED_PROMPT = "stoo_tee person, professional headshot, studio lighting, neutral background" #@param {type:"string"}
SCHED_NEGATIVE = "deformed, ugly, bad anatomy, disfigured, poorly drawn face, mutation, extra limb, bad hands, fused fingers, too many fingers, blurry, low quality" #@param {type:"string"}

#@markdown ### Settings
SCHED_SEED = 42 #@param {type:"integer"}
SCHED_STEPS = 30 #@param {type:"slider", min:10, max:50, step:5}
SCHED_CFG = 7.5 #@param {type:"slider", min:1, max:20, step:0.5}
SCHED_WIDTH = 1024 #@param {type:"slider", min:512, max:1536, step:64}
SCHED_HEIGHT = 1024 #@param {type:"slider", min:512, max:1536, step:64}

schedulers_to_test = ["DPM++ 2M Karras", "DPM++ SDE Karras", "Euler", "Euler a", "DDIM", "UniPC"]

# Build prompt
prompt = SCHED_PROMPT
if PREPEND_TRIGGER and TRIGGER_WORD and not prompt.lower().startswith(TRIGGER_WORD.lower()):
    prompt = f"{TRIGGER_WORD}, {prompt}"

# Compel embeddings
conditioning, pooled = compel(prompt)
neg_conditioning, neg_pooled = compel(SCHED_NEGATIVE)

print(f"Prompt: {prompt}")
print(f"Seed: {SCHED_SEED} | Scale: {LORA_SCALE}")
print(f"Testing {len(schedulers_to_test)} schedulers\n")

sched_images = []
for sched_name in schedulers_to_test:
    pipe.scheduler = SAMPLERS[sched_name]()
    generator = torch.Generator(device="cuda").manual_seed(SCHED_SEED)

    image = pipe(
        prompt_embeds=conditioning,
        pooled_prompt_embeds=pooled,
        negative_prompt_embeds=neg_conditioning,
        negative_pooled_prompt_embeds=neg_pooled,
        width=SCHED_WIDTH,
        height=SCHED_HEIGHT,
        guidance_scale=SCHED_CFG,
        num_inference_steps=SCHED_STEPS,
        generator=generator,
        cross_attention_kwargs={"scale": LORA_SCALE},
    ).images[0]

    sched_images.append((image, sched_name))
    print(f"  {sched_name} done")

# Display comparison grid
cols = min(len(schedulers_to_test), 3)
rows = (len(schedulers_to_test) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if rows == 1 and cols == 1:
    axes = [[axes]]
elif rows == 1:
    axes = [axes]
elif cols == 1:
    axes = [[ax] for ax in axes]

for idx, (image, sched_name) in enumerate(sched_images):
    row, col = idx // cols, idx % cols
    axes[row][col].imshow(image)
    axes[row][col].set_title(sched_name, fontsize=11, fontweight='bold')
    axes[row][col].axis('off')
for idx in range(len(schedulers_to_test), rows * cols):
    row, col = idx // cols, idx % cols
    axes[row][col].axis('off')

plt.suptitle(f"Scheduler Comparison | seed={SCHED_SEED} | scale={LORA_SCALE} | steps={SCHED_STEPS}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
#@title 8. Cleanup
#@markdown Free GPU memory when done.

import gc
import torch

try:
    del pipe, compel, vae
except NameError:
    pass

torch.cuda.empty_cache()
gc.collect()

!nvidia-smi
print("\nGPU memory cleared!")